# Demo of Optuna
This is a portion of the notebook for
### Chapter 10 – Building Neural Networks with PyTorch
**Hands-on Machine Learning with Scikit-Learn and Pytorch by Aurelien Geron, O'Reilly 2025**

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/ageron/handson-mlp/blob/main/10_neural_nets_with_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/ageron/handson-mlp/blob/main/10_neural_nets_with_pytorch.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

# Setup

This project requires Python 3.10 or above:

In [1]:
import sys

assert sys.version_info >= (3, 10)

It also requires Scikit-Learn ≥ 1.6.1:

In [2]:
from packaging.version import Version
import sklearn

assert Version(sklearn.__version__) >= Version("1.6.1")

Are we using Colab or Kaggle?

In [3]:
IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "kaggle_secrets" in sys.modules

If using Colab, a couple libraries are not pre-installed so we must install them manually:

In [4]:
if IS_COLAB:
    %pip install -q optuna torchmetrics

In [5]:
!pip install -q optuna torchmetrics

And of course we need PyTorch, specifically PyTorch ≥ 2.6.0:

In [6]:
import torch

assert Version(torch.__version__) >= Version("2.6.0")

### Harware Accelarot

In [7]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device


'cuda'

Let us define the default font sizes to make the figures prettier:

In [8]:
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

In [9]:
import torchmetrics

In [10]:
import torch.nn as nn

torch.manual_seed(42)  # to get reproducible results

# Building an Image Classifier with PyTorch

## Using TorchVision to Load the Dataset

In [11]:
import torchvision
import torchvision.transforms.v2 as T

toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

train_and_valid_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=True, download=True, transform=toTensor)
test_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=False, download=True, transform=toTensor)

torch.manual_seed(42)
train_data, valid_data = torch.utils.data.random_split(
    train_and_valid_data, [55_000, 5_000])

In [12]:
from torch.utils.data import TensorDataset, DataLoader

In [13]:
torch.manual_seed(42)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)

Each entry is a tuple (image, target):

In [14]:
X_sample, y_sample = train_data[0]

Each image has a shape \[channels, rows, columns\]. Grayscale images like in Fashion MNIST have a single channel (while RGB images have 3, and other types of images, such as satellite images, may have many more). Fashion images are grayscale and 28x28 pixels:

In [15]:
X_sample.shape

torch.Size([1, 28, 28])

In [16]:
X_sample.dtype

torch.float32

In [17]:
train_and_valid_data.classes[y_sample]

'Ankle boot'

## Building the Classifier

In [18]:
import torch.nn as nn

torch.manual_seed(42)  # to get reproducible results

In [19]:
class ImageClassifier(nn.Module):
    def __init__(self, n_inputs, n_hidden1, n_hidden2, n_classes):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_inputs, n_hidden1),
            nn.ReLU(),
            nn.Linear(n_hidden1, n_hidden2),
            nn.ReLU(),
            nn.Linear(n_hidden2, n_classes)
        )

    def forward(self, X):
        return self.mlp(X)

torch.manual_seed(42)
model = ImageClassifier(n_inputs=1 * 28 * 28, n_hidden1=300, n_hidden2=100,
                        n_classes=10).to(device)
xentropy = nn.CrossEntropyLoss()

In [20]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()  # reset the metric at the beginning
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)  # update it at each iteration
    return metric.compute()  # compute the final result at the end

In [21]:
def train2(model, optimizer, criterion, metric, train_loader, valid_loader,
               n_epochs):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.
        metric.reset()
        for X_batch, y_batch in train_loader:
            model.train()
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        mean_loss = total_loss / len(train_loader)
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_tm(model, valid_loader, metric).item())
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history

In [22]:
learning_rate = 0.4
n_epochs = 20

In [23]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
_ = train2(model, optimizer, xentropy, accuracy, train_loader, valid_loader,
           n_epochs)

Epoch 1/20, train loss: 0.6058, train metric: 0.7816, valid metric: 0.8416
Epoch 2/20, train loss: 0.4059, train metric: 0.8495, valid metric: 0.8464
Epoch 3/20, train loss: 0.3632, train metric: 0.8655, valid metric: 0.8560
Epoch 4/20, train loss: 0.3357, train metric: 0.8765, valid metric: 0.8680
Epoch 5/20, train loss: 0.3150, train metric: 0.8838, valid metric: 0.8790
Epoch 6/20, train loss: 0.2998, train metric: 0.8863, valid metric: 0.8670
Epoch 7/20, train loss: 0.2852, train metric: 0.8926, valid metric: 0.8728
Epoch 8/20, train loss: 0.2749, train metric: 0.8954, valid metric: 0.8768
Epoch 9/20, train loss: 0.2639, train metric: 0.9014, valid metric: 0.8800
Epoch 10/20, train loss: 0.2528, train metric: 0.9042, valid metric: 0.8792
Epoch 11/20, train loss: 0.2457, train metric: 0.9071, valid metric: 0.8854
Epoch 12/20, train loss: 0.2359, train metric: 0.9107, valid metric: 0.8870
Epoch 13/20, train loss: 0.2302, train metric: 0.9125, valid metric: 0.8828
Epoch 14/20, train lo

In [24]:
model.eval()
X_new, y_new = next(iter(valid_loader))
X_new = X_new[:3].to(device)
with torch.no_grad():
    y_pred_logits = model(X_new)
y_pred = y_pred_logits.argmax(dim=1)  # index of the largest logit
y_pred

tensor([7, 4, 2], device='cuda:0')

In [25]:
[train_and_valid_data.classes[index] for index in y_pred]

['Sneaker', 'Coat', 'Pullover']

Let's check whether the model made the correct predictions:

In [26]:
y_new[:3]

tensor([7, 4, 2])

All correct! 😃

In [27]:
import torch.nn.functional as F
y_proba = F.softmax(y_pred_logits, dim=1)
if device == "mps":
    y_proba = y_proba.cpu()
y_proba.round(decimals=3)

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0090, 0.0000, 0.8230, 0.0000,
         0.1670],
        [0.0000, 0.0000, 0.0030, 0.0000, 0.9960, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0020, 0.0000, 0.6330, 0.0000, 0.1560, 0.0000, 0.2060, 0.0000, 0.0020,
         0.0000]], device='cuda:0')

In [28]:
y_top4_values, y_top4_indices = torch.topk(y_pred_logits, k=4, dim=1)
y_top4_probas = F.softmax(y_top4_values, dim=1)
if device == "mps":
    y_top4_probas = y_top4_probas.cpu()
y_top4_probas.round(decimals=3)

tensor([[0.8230, 0.1670, 0.0090, 0.0000],
        [0.9960, 0.0030, 0.0000, 0.0000],
        [0.6350, 0.2070, 0.1560, 0.0020]], device='cuda:0')

In [29]:
y_top4_indices

tensor([[7, 9, 5, 8],
        [4, 2, 6, 8],
        [2, 6, 4, 0]], device='cuda:0')

In [30]:
sum([param.numel() for param in model.parameters()])

266610

# Hyperparameter Tuning using Optuna

In [31]:
import optuna

def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    n_hidden = trial.suggest_int("n_hidden", 20, 300)
    model = ImageClassifier(n_inputs=1 * 28 * 28, n_hidden1=n_hidden,
                            n_hidden2=n_hidden, n_classes=10).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    xentropy = nn.CrossEntropyLoss()
    accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10)
    accuracy = accuracy.to(device)
    history = train2(model, optimizer, xentropy, accuracy, train_loader,
                     valid_loader, n_epochs=10)
    validation_accuracy = max(history["valid_metrics"])
    return validation_accuracy

In [32]:
torch.manual_seed(42)
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=5)

[I 2026-02-11 22:21:57,579] A new study created in memory with name: no-name-0962a281-942d-4ba7-8f35-257be5cb3232


Epoch 1/10, train loss: 2.2769, train metric: 0.1471, valid metric: 0.1860
Epoch 2/10, train loss: 2.2093, train metric: 0.2794, valid metric: 0.3500
Epoch 3/10, train loss: 2.1164, train metric: 0.4110, valid metric: 0.4554
Epoch 4/10, train loss: 1.9776, train metric: 0.5137, valid metric: 0.5560
Epoch 5/10, train loss: 1.7867, train metric: 0.5826, valid metric: 0.6026
Epoch 6/10, train loss: 1.5775, train metric: 0.6184, valid metric: 0.6228
Epoch 7/10, train loss: 1.3978, train metric: 0.6288, valid metric: 0.6326
Epoch 8/10, train loss: 1.2605, train metric: 0.6360, valid metric: 0.6372
Epoch 9/10, train loss: 1.1572, train metric: 0.6467, valid metric: 0.6424


[I 2026-02-11 22:24:57,608] Trial 0 finished with value: 0.6435999870300293 and parameters: {'learning_rate': 0.00031489116479568613, 'n_hidden': 287}. Best is trial 0 with value: 0.6435999870300293.


Epoch 10/10, train loss: 1.0782, train metric: 0.6538, valid metric: 0.6436
Epoch 1/10, train loss: 1.1459, train metric: 0.6227, valid metric: 0.7338
Epoch 2/10, train loss: 0.6108, train metric: 0.7842, valid metric: 0.7996
Epoch 3/10, train loss: 0.5203, train metric: 0.8170, valid metric: 0.8092
Epoch 4/10, train loss: 0.4810, train metric: 0.8303, valid metric: 0.8308
Epoch 5/10, train loss: 0.4558, train metric: 0.8403, valid metric: 0.8354
Epoch 6/10, train loss: 0.4388, train metric: 0.8461, valid metric: 0.8446
Epoch 7/10, train loss: 0.4240, train metric: 0.8513, valid metric: 0.8410
Epoch 8/10, train loss: 0.4123, train metric: 0.8564, valid metric: 0.8510
Epoch 9/10, train loss: 0.3998, train metric: 0.8600, valid metric: 0.8530


[I 2026-02-11 22:27:48,901] Trial 1 finished with value: 0.8539999723434448 and parameters: {'learning_rate': 0.008471801418819975, 'n_hidden': 188}. Best is trial 1 with value: 0.8539999723434448.


...

Epoch 10/10, train loss: 2.2414, train metric: 0.2299, valid metric: 0.2334
Epoch 1/10, train loss: 2.3035, train metric: 0.1373, valid metric: 0.1526
Epoch 2/10, train loss: 2.3005, train metric: 0.1569, valid metric: 0.1724
Epoch 3/10, train loss: 2.2975, train metric: 0.1755, valid metric: 0.1896
Epoch 4/10, train loss: 2.2945, train metric: 0.1941, valid metric: 0.2132
Epoch 5/10, train loss: 2.2914, train metric: 0.2105, valid metric: 0.2288
Epoch 6/10, train loss: 2.2884, train metric: 0.2261, valid metric: 0.2418
Epoch 7/10, train loss: 2.2853, train metric: 0.2419, valid metric: 0.2580
Epoch 8/10, train loss: 2.2823, train metric: 0.2581, valid metric: 0.2742
Epoch 9/10, train loss: 2.2792, train metric: 0.2736, valid metric: 0.2918


[I 2026-02-11 22:33:32,437] Trial 3 finished with value: 0.30959999561309814 and parameters: {'learning_rate': 1.7073967431528103e-05, 'n_hidden': 263}. Best is trial 1 with value: 0.8539999723434448.


Epoch 10/10, train loss: 2.2761, train metric: 0.2897, valid metric: 0.3096
Epoch 1/10, train loss: 1.8379, train metric: 0.4869, valid metric: 0.6208
Epoch 2/10, train loss: 0.9751, train metric: 0.6666, valid metric: 0.6976
Epoch 3/10, train loss: 0.7608, train metric: 0.7253, valid metric: 0.7416
Epoch 4/10, train loss: 0.6704, train metric: 0.7639, valid metric: 0.7720
Epoch 5/10, train loss: 0.6108, train metric: 0.7913, valid metric: 0.7906
Epoch 6/10, train loss: 0.5687, train metric: 0.8053, valid metric: 0.8050
Epoch 7/10, train loss: 0.5385, train metric: 0.8165, valid metric: 0.8082
Epoch 8/10, train loss: 0.5158, train metric: 0.8243, valid metric: 0.8216
Epoch 9/10, train loss: 0.4988, train metric: 0.8281, valid metric: 0.8220


[I 2026-02-11 22:36:24,365] Trial 4 finished with value: 0.8220000267028809 and parameters: {'learning_rate': 0.002537815508265664, 'n_hidden': 218}. Best is trial 1 with value: 0.8539999723434448.


Epoch 10/10, train loss: 0.4842, train metric: 0.8330, valid metric: 0.8092


In [33]:
study.best_params

{'learning_rate': 0.008471801418819975, 'n_hidden': 188}

In [34]:
study.best_value

0.8539999723434448

In [35]:
def objective(trial, train_loader, valid_loader):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    n_hidden = trial.suggest_int("n_hidden", 20, 300)
    model = ImageClassifier(n_inputs=1 * 28 * 28, n_hidden1=n_hidden,
                            n_hidden2=n_hidden, n_classes=10).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    xentropy = nn.CrossEntropyLoss()
    accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10)
    accuracy = accuracy.to(device)
    best_validation_accuracy = 0.0
    for epoch in range(n_epochs):
        history = train2(model, optimizer, xentropy, accuracy, train_loader,
                         valid_loader, n_epochs=1)
        validation_accuracy = max(history["valid_metrics"])
        if validation_accuracy > best_validation_accuracy:
            best_validation_accuracy = validation_accuracy
        trial.report(validation_accuracy, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return best_validation_accuracy

In [36]:
objective_with_data = lambda trial: objective(
    trial, train_loader=train_loader, valid_loader=valid_loader)

In [37]:
from functools import partial

objective_with_data = partial(objective, train_loader=train_loader,
                              valid_loader=valid_loader)

In [38]:
torch.manual_seed(42)
sampler = optuna.samplers.TPESampler(seed=42)
pruner = optuna.pruners.MedianPruner()
study = optuna.create_study(direction="maximize", sampler=sampler,
                            pruner=pruner)
study.optimize(objective_with_data, n_trials=20)

[I 2026-02-11 22:36:24,499] A new study created in memory with name: no-name-2879029d-eed2-40b1-b326-d2bf7ab4ae51


Epoch 1/1, train loss: 2.2769, train metric: 0.1471, valid metric: 0.1860
Epoch 1/1, train loss: 2.2093, train metric: 0.2794, valid metric: 0.3500
Epoch 1/1, train loss: 2.1164, train metric: 0.4110, valid metric: 0.4554
Epoch 1/1, train loss: 1.9776, train metric: 0.5137, valid metric: 0.5560
Epoch 1/1, train loss: 1.7867, train metric: 0.5826, valid metric: 0.6026
Epoch 1/1, train loss: 1.5775, train metric: 0.6184, valid metric: 0.6228
Epoch 1/1, train loss: 1.3978, train metric: 0.6288, valid metric: 0.6326
Epoch 1/1, train loss: 1.2605, train metric: 0.6360, valid metric: 0.6372
Epoch 1/1, train loss: 1.1572, train metric: 0.6467, valid metric: 0.6424
Epoch 1/1, train loss: 1.0782, train metric: 0.6538, valid metric: 0.6436
Epoch 1/1, train loss: 1.0162, train metric: 0.6611, valid metric: 0.6530
Epoch 1/1, train loss: 0.9665, train metric: 0.6689, valid metric: 0.6620
Epoch 1/1, train loss: 0.9258, train metric: 0.6761, valid metric: 0.6700
Epoch 1/1, train loss: 0.8919, train m

[I 2026-02-11 22:42:02,649] Trial 0 finished with value: 0.7089999914169312 and parameters: {'learning_rate': 0.00031489116479568613, 'n_hidden': 287}. Best is trial 0 with value: 0.7089999914169312.


Epoch 1/1, train loss: 0.7647, train metric: 0.7196, valid metric: 0.7082
Epoch 1/1, train loss: 1.1485, train metric: 0.6157, valid metric: 0.7330
Epoch 1/1, train loss: 0.6133, train metric: 0.7863, valid metric: 0.8084
Epoch 1/1, train loss: 0.5200, train metric: 0.8179, valid metric: 0.8134
Epoch 1/1, train loss: 0.4783, train metric: 0.8310, valid metric: 0.8236
Epoch 1/1, train loss: 0.4532, train metric: 0.8401, valid metric: 0.8032
Epoch 1/1, train loss: 0.4357, train metric: 0.8464, valid metric: 0.8446
Epoch 1/1, train loss: 0.4209, train metric: 0.8511, valid metric: 0.8278
Epoch 1/1, train loss: 0.4082, train metric: 0.8561, valid metric: 0.8400
Epoch 1/1, train loss: 0.3981, train metric: 0.8604, valid metric: 0.8522
Epoch 1/1, train loss: 0.3882, train metric: 0.8642, valid metric: 0.8584
Epoch 1/1, train loss: 0.3783, train metric: 0.8663, valid metric: 0.8536
Epoch 1/1, train loss: 0.3698, train metric: 0.8695, valid metric: 0.8570
Epoch 1/1, train loss: 0.3631, train m

[I 2026-02-11 22:47:52,162] Trial 1 finished with value: 0.8676000237464905 and parameters: {'learning_rate': 0.008471801418819975, 'n_hidden': 188}. Best is trial 1 with value: 0.8676000237464905.


...

Epoch 1/1, train loss: 1.5604, train metric: 0.5039, valid metric: 0.6536
Epoch 1/1, train loss: 0.6189, train metric: 0.7751, valid metric: 0.8314
Epoch 1/1, train loss: 0.4237, train metric: 0.8447, valid metric: 0.8492
Epoch 1/1, train loss: 0.3815, train metric: 0.8597, valid metric: 0.8544
Epoch 1/1, train loss: 0.3566, train metric: 0.8677, valid metric: 0.8592
Epoch 1/1, train loss: 0.3376, train metric: 0.8742, valid metric: 0.8660
Epoch 1/1, train loss: 0.3241, train metric: 0.8798, valid metric: 0.8426
Epoch 1/1, train loss: 0.3118, train metric: 0.8843, valid metric: 0.8708
Epoch 1/1, train loss: 0.3020, train metric: 0.8869, valid metric: 0.8700
Epoch 1/1, train loss: 0.2928, train metric: 0.8897, valid metric: 0.8772
Epoch 1/1, train loss: 0.2856, train metric: 0.8932, valid metric: 0.8762
Epoch 1/1, train loss: 0.2774, train metric: 0.8957, valid metric: 0.8746
Epoch 1/1, train loss: 0.2716, train metric: 0.8971, valid metric: 0.8648
Epoch 1/1, train loss: 0.2652, train m

[I 2026-02-11 23:48:52,300] Trial 18 finished with value: 0.8845999836921692 and parameters: {'learning_rate': 0.09548128419071349, 'n_hidden': 50}. Best is trial 12 with value: 0.8880000114440918.


Epoch 1/1, train loss: 0.2350, train metric: 0.9119, valid metric: 0.8846


[I 2026-02-11 23:49:11,931] Trial 19 pruned. 


Epoch 1/1, train loss: 0.7417, train metric: 0.7394, valid metric: 0.7642


In [39]:
study.best_value

0.8880000114440918

In [40]:
study.best_params

{'learning_rate': 0.08525846269447779, 'n_hidden': 116}